In [1]:
# Reading .sql files from "../sql_datasets/datasets/sql-server/" supporting T-SQl

In [6]:
from pathlib import Path
import re
from sqlalchemy import create_engine, text

# 1) point to your SQL Server .sql file
src = Path("../sql_datasets/datasets/sql-server/init-sqlserver-salesdb.sql").read_text()

# 2) basic T-SQL -> SQLite normalization
def tsql_to_sqlite(sql: str) -> str:
    # remove BOM / CRLF
    s = sql.replace("\r\n", "\n")
    # remove comments like lines starting with --
    # (keep block comments if you want; they are harmless)
    # s = re.sub(r'--.*', '', s)

    # remove batch and db commands unsupported in SQLite
    s = re.sub(r'(?im)^\s*USE\s+.+?;\s*', '', s)
    s = re.sub(r'(?im)^\s*GO\s*$', '', s)          # standalone GO lines
    s = re.sub(r'(?im)^\s*EXEC\s+sys\.sp_executesql.+?;\s*', '', s)
    s = re.sub(r'(?is)IF\s+EXISTS\s*\(.*?DROP\s+DATABASE.*?END;\s*', '', s)
    s = re.sub(r'(?im)^\s*CREATE\s+DATABASE\s+.+?;\s*', '', s)
    s = re.sub(r'(?im)^\s*ALTER\s+DATABASE\s+.+?;\s*', '', s)

    # drop schema prefixes like Sales.Table -> Table
    s = re.sub(r'(?<!\w)Sales\.', '', s)

    # type mappings
    s = re.sub(r'(?i)\bDATETIME2\b', 'TEXT', s)
    s = re.sub(r'(?i)\bDATETIME\b',  'TEXT', s)
    s = re.sub(r'(?i)\bDATE\b',      'TEXT', s)
    s = re.sub(r'(?i)\bNVARCHAR\(\s*MAX\s*\)', 'TEXT', s)
    s = re.sub(r'(?i)\bVARCHAR\(\s*MAX\s*\)',  'TEXT', s)
    s = re.sub(r'(?i)\bNVARCHAR\(\d+\)', 'TEXT', s)
    s = re.sub(r'(?i)\bVARCHAR\(\d+\)',  'TEXT', s)
    s = re.sub(r'(?i)\bCHAR\(\d+\)',     'TEXT', s)

    # identity -> INTEGER PRIMARY KEY AUTOINCREMENT (only if it's the PK)
    s = re.sub(r'(?i)\bIDENTITY\s*\(\s*\d+\s*,\s*\d+\s*\)', '', s)

    # T-SQL PK names -> simple PRIMARY KEY (SQLite ignores named constraints anyway)
    s = re.sub(r'(?i)CONSTRAINT\s+\w+\s+PRIMARY\s+KEY', 'PRIMARY KEY', s)

    # quote the Sales column name to avoid confusion
    s = re.sub(r'(?i)\bSales\s+INT\b', '"Sales" INT', s)

    # remove empty lines of GO or leftovers
    s = re.sub(r'\n{2,}', '\n', s).strip()
    return s

normalized = tsql_to_sqlite(src)

# 3) split statements on semicolon that ends a statement (basic splitter)
stmts = [st.strip() for st in normalized.split(';') if st.strip()]

# 4) Prepend DROP TABLE IF EXISTS for the tables we know, to re-run cleanly
drops = [
    "DROP TABLE IF EXISTS Customers",
    "DROP TABLE IF EXISTS Employees",
    "DROP TABLE IF EXISTS Products",
    "DROP TABLE IF EXISTS Orders",
    "DROP TABLE IF EXISTS OrdersArchive"
]

engine = create_engine("sqlite:///portfolio.db")

with engine.begin() as conn:
    for d in drops:
        conn.execute(text(d))
    for i, st in enumerate(stmts, 1):
        try:
            conn.execute(text(st))
        except Exception as e:
            print(f"[WARN] Statement #{i} failed:\n{st[:200]}...\nError: {e}\n")
            # You can choose to raise to stop:
            # raise

print("✅ Converted script executed on SQLite (portfolio.db).")



[WARN] Statement #1 failed:
/*
Database Creation and Table Setup Script
Script Purpose:
    This script ...
Error: (sqlite3.OperationalError) near "IF": syntax error
[SQL: /*
Database Creation and Table Setup Script
Script Purpose:
    This script creates a new SQL Server database named 'SalesDB'. 
    If the database already exists, it is dropped to ensure a clean setup. 
    The script then creates three tables: 'customers', 'orders', and 'employees' 
    with their respective schemas, and populates them with sample data.
    
    Running this script will drop the entire 'SalesDB' database if it exists, 
    permanently deleting all data within it. Proceed with caution and ensure you 
    have proper backups before executing this script.
*/
-- Drop and recreate the 'SalesDB' database
-- Create the 'SalesDB' database
-- Check if the schema 'Sales' exists
IF EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = 'Sales')
BEGIN
    -- If it does exist, drop the 'Sales' sc

In [10]:
%load_ext sql
%sql sqlite:///portfolio.db
%config SqlMagic.autopandas = True


The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [11]:
%%sql
SELECT * FROM Customers
LIMIT 5;

 * sqlite:///portfolio.db
Done.


,CustomerID,FirstName,LastName,Country,Score
0,1,Jossef,Goldberg,Germany,350.0
1,2,Kevin,Brown,USA,900.0
2,3,Mary,None,USA,750.0
3,4,Mark,Schwarz,Germany,500.0
4,5,Anna,Adams,USA,NaN


In [14]:
# --- SQL Server (.sql) -> SQLite loader (self-contained) --------------------
from pathlib import Path
import re
from typing import Iterable
from sqlalchemy import create_engine, text
import pandas as pd

# 0) point SQLite engine at your portfolio db (created if it doesn't exist)
engine = create_engine("sqlite:///portfolio.db")

# 1) robust T-SQL -> SQLite normalizer
def tsql_to_sqlite(sql: str, schemas: Iterable[str] = ("dbo", "Sales")) -> str:
    s = sql.replace("\r\n", "\n")

    # strip block comments (/* ... */) but leave inline content if any
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.S)

    # remove USE/CREATE/ALTER/RESTORE DB & batch separators (SQLite doesn't support)
    s = re.sub(r"(?im)^\s*USE\s+.+?;\s*$", "", s)
    s = re.sub(r"(?im)^\s*GO\]?[\s;]*$", "", s)  # GO or GO]
    s = re.sub(r"(?im)^\s*CREATE\s+DATABASE\b.+?;\s*$", "", s)
    s = re.sub(r"(?im)^\s*ALTER\s+DATABASE\b.+?;\s*$", "", s)
    s = re.sub(r"(?is)IF\s+EXISTS\s*\(.*?DROP\s+DATABASE.*?END\s*;\s*", "", s)
    s = re.sub(r"(?im)^\s*EXEC\s+sys\.sp_executesql.+?;\s*$", "", s)

    # DROP/CREATE SCHEMA not supported; drop them
    s = re.sub(r"(?im)^\s*DROP\s+SCHEMA\b.+?;\s*$", "", s)
    s = re.sub(r"(?im)^\s*CREATE\s+SCHEMA\b.+?;\s*$", "", s)

    # remove bracketed identifiers [LikeThis] -> LikeThis
    s = re.sub(r"\[([^\]]+)\]", r"\1", s)

    # strip known schema prefixes: dbo.Table, Sales.Table -> Table
    if schemas:
        schema_pat = r"(?<!\w)(?:{})(?:\.)".format("|".join([re.escape(sc) for sc in schemas]))
        s = re.sub(schema_pat, "", s, flags=re.I)

    # type mappings (SQLite is dynamic, but we coerce to common types)
    s = re.sub(r"(?i)\bDATETIME2\b", "TEXT", s)
    s = re.sub(r"(?i)\bSMALLDATETIME\b", "TEXT", s)
    s = re.sub(r"(?i)\bDATETIME\b", "TEXT", s)
    s = re.sub(r"(?i)\bDATE\b", "TEXT", s)
    s = re.sub(r"(?i)\bTIME\b", "TEXT", s)
    s = re.sub(r"(?i)\bBIT\b", "INTEGER", s)

    s = re.sub(r"(?i)\bNVARCHAR\(\s*MAX\s*\)", "TEXT", s)
    s = re.sub(r"(?i)\bVARCHAR\(\s*MAX\s*\)", "TEXT", s)
    s = re.sub(r"(?i)\bNVARCHAR\(\d+\)", "TEXT", s)
    s = re.sub(r"(?i)\bVARCHAR\(\d+\)", "TEXT", s)
    s = re.sub(r"(?i)\bCHAR\(\d+\)", "TEXT", s)

    # identity -> remove (SQLite autoincrement handled by INTEGER PRIMARY KEY)
    s = re.sub(r"(?i)\bIDENTITY\s*\(\s*\d+\s*,\s*\d+\s*\)", "", s)

    # named PK constraints -> plain PRIMARY KEY
    s = re.sub(r"(?i)CONSTRAINT\s+\w+\s+PRIMARY\s+KEY", "PRIMARY KEY", s)

    # tiny cleanup of stray semicolons + blank lines
    s = re.sub(r"\n{2,}", "\n", s).strip()
    return s

# 2) execute a .sql file (with conversion) statement-by-statement
def run_sqlserver_file_to_sqlite(path: str):
    raw = Path(path).read_text()
    norm = tsql_to_sqlite(raw)

    # naive split on ';' that are statement terminators (good enough for DDL/DML here)
    stmts = [st.strip() for st in norm.split(";") if st.strip()]
    executed, skipped = 0, 0

    with engine.begin() as conn:
        for st in stmts:
            try:
                conn.execute(text(st))
                executed += 1
            except Exception as e:
                skipped += 1
                print(f"[WARN] Skipping statement due to error:\n{st[:200]}...\n -> {e}\n")

    print(f"✅ {path} --> executed {executed} statements, skipped {skipped}")

# 3) list your SQL Server scripts here (add more files as needed)
sqlserver_files = [
    "../sql_datasets/datasets/sql-server/init-sqlserver-salesdb.sql",
    "../sql_datasets/datasets/sql-server/init-sqlserver-mydatabase.sql",
]

# 4) (optional) clean tables before loading (safe to re-run notebooks)
def drop_if_exists(conn, tables):
    for t in tables:
        try:
            conn.execute(text(f"DROP TABLE IF EXISTS {t}"))
        except Exception:
            pass

with engine.begin() as c:
    # if you know likely table names, drop them to make runs idempotent
    drop_if_exists(c, ["Customers","Employees","Products","Orders","OrdersArchive",
                       "Customer","CustomerAddress","CustomerOrders","SalesDB","CustomerDB"])

# 5) run all files
for f in sqlserver_files:
    run_sqlserver_file_to_sqlite(f)

# 6) verify: list tables, preview a few rows from each if present
with engine.connect() as conn:
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn)
tables


[WARN] Skipping statement due to error:
-- Drop and recreate the 'SalesDB' database
-- Create the 'SalesDB' database
-- Check if the schema 'Sales' exists
IF EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = 'Sales')
BEG...
 -> (sqlite3.OperationalError) near "IF": syntax error
[SQL: -- Drop and recreate the 'SalesDB' database
-- Create the 'SalesDB' database
-- Check if the schema 'Sales' exists
IF EXISTS (SELECT 1 FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = 'Sales')
BEGIN
    -- If it does exist, drop the 'Sales' schema
END]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

✅ ../sql_datasets/datasets/sql-server/init-sqlserver-salesdb.sql --> executed 10 statements, skipped 1
✅ ../sql_datasets/datasets/sql-server/init-sqlserver-mydatabase.sql --> executed 6 statements, skipped 0


,name
0,Employees
1,OrdersArchive
2,Products
3,customers
4,orders


In [15]:
import pandas as pd
from sqlalchemy import create_engine
engine = create_engine("sqlite:///portfolio.db")

def peek(table, n=5):
    try:
        return pd.read_sql(f"SELECT * FROM {table} LIMIT {n};", engine)
    except Exception as e:
        print(f"{table}: {e}")

for t in ["Customers","Employees","Products","Orders","OrdersArchive"]:
    df = peek(t, 5)
    if df is not None:
        print(f"\n=== {t} (top {len(df)} rows) ===")
        display(df)



=== Customers (top 5 rows) ===


,id,first_name,country,score
0,1,Maria,Germany,350
1,2,John,USA,900
2,3,Georg,UK,750
3,4,Martin,Germany,500
4,5,Peter,USA,0



=== Employees (top 5 rows) ===


,EmployeeID,FirstName,LastName,Department,BirthDate,Gender,Salary,ManagerID
0,1,Frank,Lee,Marketing,1988-12-05,M,55000,NaN
1,2,Kevin,Brown,Marketing,1972-11-25,M,65000,1.0
2,3,Mary,None,Sales,1986-01-05,F,75000,1.0
3,4,Michael,Ray,Sales,1977-02-10,M,90000,2.0
4,5,Carol,Baker,Sales,1982-02-11,F,55000,3.0



=== Products (top 5 rows) ===


,ProductID,Product,Category,Price
0,101,Bottle,Accessories,10
1,102,Tire,Accessories,15
2,103,Socks,Clothing,20
3,104,Caps,Clothing,25
4,105,Gloves,Clothing,30



=== Orders (top 4 rows) ===


,order_id,customer_id,order_date,sales
0,1001,1,2021-01-11,35
1,1002,2,2021-04-05,15
2,1003,3,2021-06-18,20
3,1004,6,2021-08-31,10



=== OrdersArchive (top 5 rows) ===


,OrderID,ProductID,CustomerID,SalesPersonID,OrderDate,ShipDate,OrderStatus,ShipAddress,BillAddress,Quantity,Sales,CreationTime
0,1,101,2,3,2024-04-01,2024-04-05,Shipped,123 Main St,456 Billing St,1,10,2024-04-01T12:34:56
1,2,102,3,3,2024-04-05,2024-04-10,Shipped,456 Elm St,789 Billing St,1,15,2024-04-05T23:22:04
2,3,101,1,4,2024-04-10,2024-04-25,Shipped,789 Maple St,789 Maple St,2,20,2024-04-10T18:24:08
3,4,105,1,3,2024-04-20,2024-04-25,Shipped,987 Victory Lane,,2,60,2024-04-20T05:50:33
4,4,105,1,3,2024-04-20,2024-04-25,Delivered,987 Victory Lane,,2,60,2024-04-20T14:50:33
